# 05 – ML Availability Model

Train and evaluate Logistic Regression and XGBoost classifiers that
predict the probability of a findable between-lines option at any
build-up event.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import load_competition
from src.features.buildup import filter_buildup
from src.features.line_detection import detect_opponent_lines
from src.features.receiver import detect_receiver_candidates
from src.features.lane import compute_lane_features
from src.features.findability import compute_findability
from src.metrics.outputs import merge_event_scores
from src.models.availability import AvailabilityModel, FEATURE_COLUMNS


## Build the modelling dataset (Euro 2020)


In [ ]:
events, frames, lineups, matches = load_competition(competition_id=55, season_id=43)
buildup_df    = filter_buildup(events)
line_df       = detect_opponent_lines(frames, buildup_df)
receivers_df  = detect_receiver_candidates(frames, buildup_df, line_df)
candidates_df = compute_lane_features(receivers_df, frames, buildup_df)
event_scores, _ = compute_findability(candidates_df)
merged_df     = merge_event_scores(buildup_df, event_scores, line_df)

model_df = merged_df.dropna(subset=['findable_option_available'])
print(f'Modelling dataset: {len(model_df):,} rows')
print(f'Class balance: {model_df["findable_option_available"].value_counts().to_dict()}')


## Train models


In [ ]:
model = AvailabilityModel()
model.fit(model_df, model='both')


## Cross-validated evaluation


In [ ]:
metrics = model.evaluate(model_df, plot=True)
for k, v in metrics.items():
    print(f'  {k}: {v:.3f}')
plt.show()


## Feature importance (XGBoost)


In [ ]:
importance_df = model.feature_importance()
importance_df.head(15).set_index('feature')['importance'].sort_values().plot(
    kind='barh', title='XGBoost Feature Importance', figsize=(8, 6)
)
plt.tight_layout()
plt.show()


## Add predicted probability to the analysis table


In [ ]:
model_df = model_df.copy()
model_df['availability_prob_xgb'] = model.predict_proba(model_df, model='xgb')
model_df['availability_prob_lr']  = model.predict_proba(model_df, model='lr')

# Top events with highest predicted probability of having a findable option
cols = ['team', 'player', 'type', 'x', 'y', 'availability_prob_xgb', 'findable_option_available']
available_cols = [c for c in cols if c in model_df.columns]
display(model_df.nlargest(10, 'availability_prob_xgb')[available_cols])
